# 🎙️ Personal AI Audiobook Creator

Creating a high-quality audiobook with your own voice is now incredibly simple. This notebook uses **zero-shot voice cloning** to narrate any EPUB file using just a few seconds of reference audio.

### 🌍 Language Support
Note: While the sample used here is in **Dutch**, this system works seamlessly for **English** and many other languages.

### 🚀 How it Works
1. **Setup:** Install the core engines.
2. **Source:** Load your EPUB (Sample provided below).
3. **Voice:** Provide a 10-second clip of the voice you want to clone.
4. **Generate:** Let the AI narrate the book in parallel.

### 🔗 Credits & Resources
- **TTS Engine:** [OmniVoice](https://github.com/k2-fsa/OmniVoice) by k2-fsa.
- **Text Processing:** `ebooklib` & `BeautifulSoup`.
- **Source Material:** [Project Gutenberg](https://www.gutenberg.org/) for the public domain text.
- **Sample Book:** [Jules Verne - De Reis naar de Maan](https://github.com/Toon-nooT/notebooks/raw/refs/heads/main/assets/De%20Reis%20naar%20de%20Maan%20in%2028%20dagen%20en%2012%20uren%20by%20Jules%20Verne.epub)

In [ ]:
# @title 1. Install Dependencies
!pip install -q ebooklib beautifulsoup4 num2words omnivoice
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
from google.colab import files
import re, os, subprocess, time, shutil
import soundfile as sf
import torch
from omnivoice import OmniVoice
from concurrent.futures import ThreadPoolExecutor

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# @title 2. Load the EPUB
# You can download the sample book here:
# https://github.com/Toon-nooT/notebooks/raw/refs/heads/main/assets/De%20Reis%20naar%20de%20Maan%20in%2028%20dagen%20en%2012%20uren%20by%20Jules%20Verne.epub

print("Upload your EPUB file:")
uploaded = files.upload()
epub_filename = list(uploaded.keys())[0]

def extract_formatted_text(item):
    soup = BeautifulSoup(item.get_content(), 'html.parser')
    return soup.get_text(separator='\n', strip=True)

book = epub.read_epub(epub_filename)
chapters_text = []

for item in book.get_items():
    if item.get_type() == ebooklib.ITEM_DOCUMENT:
        text = extract_formatted_text(item)
        if len(text.strip()) > 100: # Filter out empty/meta pages
            chapters_text.append(text)

print(f"\nSuccessfully extracted {len(chapters_text)} chapters.")

Upload your EPUB file:


Saving De Reis naar de Maan in 28 dagen en 12 uren by Jules Verne.epub to De Reis naar de Maan in 28 dagen en 12 uren by Jules Verne (2).epub

Successfully extracted 8 chapters.


In [ ]:
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

# 1. Improved extraction function that preserves newlines
def extract_formatted_text(item):
    soup = BeautifulSoup(item.get_content(), 'html.parser')
    # Use newline as separator to keep paragraph structure
    return soup.get_text(separator='\n', strip=True)

# 2. Re-process the book
book = epub.read_epub(epub_filename)
extracted_data = []

for item in book.get_items():
    if item.get_type() == ebooklib.ITEM_DOCUMENT:
        text = extract_formatted_text(item)
        # Using a lower threshold to ensure we don't miss short chapters
        if len(text.strip()) > 50:
            # Store both the item name/ID and the content for verification
            extracted_data.append({
                'id': item.get_id(),
                'name': item.get_name(),
                'content': text
            })

chapters_text = [d['content'] for d in extracted_data]

print(f"Total items extracted: {len(extracted_data)}")


Total items extracted: 8


In [ ]:
# '123' becomes 'one hundred twenty-three') using the num2words library.

import re
from num2words import num2words

def normalize_text_numbers(text, lang='nl'):
    """Finds numbers in text and converts them to words."""
    def replace(match):
        number_str = match.group()
        try:
            return num2words(int(number_str), lang=lang)
        except:
            return number_str

    # Matches sequences of digits
    return re.sub(r'\d+', replace, text)

# Update the processing logic to include normalization
def process_chapter_with_normalization(chapter_data):
    global completed_count
    idx, chapter_content = chapter_data
    chapter_start = time.time()

    # Normalize numbers to Dutch words
    normalized_content = normalize_text_numbers(chapter_content, lang='nl')

    lines = normalized_content.split('\n')
    title_line = lines[0]
    final_text = title_line + "\n\n\n" + "\n".join(lines[1:])

    safe_title = sanitize_filename(title_line)
    base_name = f"{BOOK_PREFIX}.{idx+1:03d}.{safe_title}"
    wav_path = os.path.join(OUTPUT_DIR, f"{base_name}.wav")
    mp3_path = os.path.join(OUTPUT_DIR, f"{base_name}.mp3")

    try:
        audio = model.generate(
            text=final_text,
            instruct="male, teenager",
            ref_audio=ref_audio_path,
            ref_text=ref_text
        )

        sf.write(wav_path, audio[0], 24000)
        subprocess.run(['ffmpeg', '-y', '-i', wav_path, '-ac', '1', '-ar', '24000', '-b:a', '64k', mp3_path], check=True, capture_output=True)

        if os.path.exists(wav_path): os.remove(wav_path)

        completed_count += 1
        elapsed = time.time() - start_time
        avg = elapsed / completed_count
        eta = ((len(chapters_text) - completed_count) * avg) / 60
        print(f"[{completed_count}/{len(chapters_text)}] Normalized & Generated: {base_name} | ETA: {eta:.1f}m")

    except Exception as e:
        print(f"Error processing {base_name}: {e}")

In [ ]:
# 3. Summary of first and last 10 chapters
def print_summary(data, count=10):
    print(f"--- FIRST {count} CHAPTERS ---")
    for i, item in enumerate(data[:count]):
        preview = item['content'].split('\n')[0][:50] # Get first line preview
        print(f"{i+1}. [{item['name']}] -> {preview}")

    print(f"\n--- LAST {count} CHAPTERS ---")
    start_idx = max(0, len(data) - count)
    for i, item in enumerate(data[start_idx:]):
        preview = item['content'].split('\n')[0][:50]
        print(f"{start_idx + i + 1}. [{item['name']}] -> {preview}")

print_summary(extracted_data)

--- FIRST 10 CHAPTERS ---
1. [2698524604637653331_27309-h-0.htm.xhtml] -> The Project Gutenberg eBook of
2. [2698524604637653331_27309-h-1.htm.xhtml] -> Eerste hoofdstuk.
3. [2698524604637653331_27309-h-2.htm.xhtml] -> Achttiende hoofdstuk.
4. [2698524604637653331_27309-h-3.htm.xhtml] -> Drieëndertigste hoofdstuk.
5. [2698524604637653331_27309-h-4.htm.xhtml] -> Zesenveertigste hoofdstuk.
6. [2698524604637653331_27309-h-5.htm.xhtml] -> Inhoudsopgave
7. [2698524604637653331_27309-h-6.htm.xhtml] -> *** END OF THE PROJECT GUTENBERG EBOOK DE REIS NAA
8. [toc.xhtml] -> Wonderreizen.

--- LAST 10 CHAPTERS ---
1. [2698524604637653331_27309-h-0.htm.xhtml] -> The Project Gutenberg eBook of
2. [2698524604637653331_27309-h-1.htm.xhtml] -> Eerste hoofdstuk.
3. [2698524604637653331_27309-h-2.htm.xhtml] -> Achttiende hoofdstuk.
4. [2698524604637653331_27309-h-3.htm.xhtml] -> Drieëndertigste hoofdstuk.
5. [2698524604637653331_27309-h-4.htm.xhtml] -> Zesenveertigste hoofdstuk.
6. [2698524604637653331_2

In [ ]:
import numpy as np

# Slicing the list to keep chapters 1 to 5
chapters_text = chapters_text[1:5]

print(f"New chapter count: {len(chapters_text)}")
print(f"First chapter in list: {chapters_text[0][:50]}...")
print(f"Last chapter in list: {chapters_text[-1][:50]}...")

New chapter count: 4
First chapter in list: Eerste hoofdstuk.
De Gun-club.
Amerika is een groo...
Last chapter in list: Zesenveertigste hoofdstuk.
Maston ten tooneele.
Er...


In [ ]:
# check content of a specific chapter
print(f"Preview of Chapter 1: {chapters_text[3][:200]}...")

Preview of Chapter 1: Zesenveertigste hoofdstuk.
Maston ten tooneele.
Er was een groote beweging aan boord van de Susquehanna. Officieren en matrozen vergaten het vreeslijk gevaar dat zij hadden
geloopen, verbrijzeld en na...


In [ ]:
# @title 3. Initialize Voice Engine
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=False,
)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

### 3.1 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

💡 PRO TIP: To record your own voice, you can use free tools like
Audacity (Desktop) or an online voice recorder (like online-voice-recorder.com).

Simply record yourself reading the text below, save it as a .wav file, and upload it here.

*"Typisch, zuchtte de vrolijke wiskundeleraar, terwijl hij naar het grijze schoolbord wees."*

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

# Note: These sentences are phonetically optimized to cover a wide range
# of sounds, which helps the model create a more accurate voice clone.
ref_text = "Typisch, zuchtte de vrolijke wiskundeleraar, terwijl hij naar het grijze schoolbord wees."

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

Upload a reference audio file (wav/mp3/flac):


Saving Record (online-audio-converter.com).wav to Record (online-audio-converter.com) (2).wav
Uploaded: Record (online-audio-converter.com) (2).wav


In [ ]:
# @title 4. Generate Audiobook (Parallel Processing)
BOOK_PREFIX = "Jules Verne - De Reis naar de Maan"
OUTPUT_DIR = "output_chapters"
THREADS = 2 # Adjust based on GPU memory
os.makedirs(OUTPUT_DIR, exist_ok=True)

def sanitize_filename(text):
    return re.sub(r'[^a-zA-Z0-9]', '_', text)[:50]

def process_chapter(chapter_data):
    idx, content = chapter_data
    title = content.split('\n')[0]
    safe_name = f"{idx:03d}_{sanitize_filename(title)}"
    wav_path = os.path.join(OUTPUT_DIR, f"{safe_name}.wav")
    mp3_path = os.path.join(OUTPUT_DIR, f"{safe_name}.mp3")

    try:
        # Using 'male' as a valid instruction item instead of 'natural narration'
        audio = model.generate(text=content, instruct="male", ref_audio=ref_audio_path, ref_text=ref_text)
        sf.write(wav_path, audio[0], 24000)
        subprocess.run(['ffmpeg', '-y', '-i', wav_path, '-b:a', '64k', mp3_path], capture_output=True)
        if os.path.exists(wav_path): os.remove(wav_path)
        print(f"Done: {safe_name}")
    except Exception as e: print(f"Error in {safe_name}: {e}")

print(f"Starting generation...")
with ThreadPoolExecutor(max_workers=THREADS) as executor:
    executor.map(process_chapter, list(enumerate(chapters_text)))

shutil.make_archive("audiobook_package", 'zip', OUTPUT_DIR)
files.download("audiobook_package.zip")

Starting generation...


In [ ]:
# 3. Summary of first and last 10 chapters
def print_summary(data, count=10):
    print(f"--- FIRST {count} CHAPTERS ---")
    for i, item in enumerate(data[:count]):
        preview = item['content'].split('\n')[0][:50] # Get first line preview
        print(f"{i+1}. [{item['name']}] -> {preview}")

    print(f"\n--- LAST {count} CHAPTERS ---")
    start_idx = max(0, len(data) - count)
    for i, item in enumerate(data[start_idx:]):
        preview = item['content'].split('\n')[0][:50]
        print(f"{start_idx + i + 1}. [{item['name']}] -> {preview}")

print_summary(extracted_data)

--- FIRST 10 CHAPTERS ---
1. [2698524604637653331_27309-h-0.htm.xhtml] -> The Project Gutenberg eBook of
2. [2698524604637653331_27309-h-1.htm.xhtml] -> Eerste hoofdstuk.
3. [2698524604637653331_27309-h-2.htm.xhtml] -> Achttiende hoofdstuk.
4. [2698524604637653331_27309-h-3.htm.xhtml] -> Drieëndertigste hoofdstuk.
5. [2698524604637653331_27309-h-4.htm.xhtml] -> Zesenveertigste hoofdstuk.
6. [2698524604637653331_27309-h-5.htm.xhtml] -> Inhoudsopgave
7. [2698524604637653331_27309-h-6.htm.xhtml] -> *** END OF THE PROJECT GUTENBERG EBOOK DE REIS NAA
8. [toc.xhtml] -> Wonderreizen.

--- LAST 10 CHAPTERS ---
1. [2698524604637653331_27309-h-0.htm.xhtml] -> The Project Gutenberg eBook of
2. [2698524604637653331_27309-h-1.htm.xhtml] -> Eerste hoofdstuk.
3. [2698524604637653331_27309-h-2.htm.xhtml] -> Achttiende hoofdstuk.
4. [2698524604637653331_27309-h-3.htm.xhtml] -> Drieëndertigste hoofdstuk.
5. [2698524604637653331_27309-h-4.htm.xhtml] -> Zesenveertigste hoofdstuk.
6. [2698524604637653331_2

In [ ]:
import numpy as np

# Slicing the list to keep chapters 1 to 5
chapters_text = chapters_text[1:5]

print(f"New chapter count: {len(chapters_text)}")
print(f"First chapter in list: {chapters_text[0][:50]}...")
print(f"Last chapter in list: {chapters_text[-1][:50]}...")

New chapter count: 4
First chapter in list: Eerste hoofdstuk.
De Gun-club.
Amerika is een groo...
Last chapter in list: Zesenveertigste hoofdstuk.
Maston ten tooneele.
Er...


In [ ]:
# check content of a specific chapter
print(f"Preview of Chapter 1: {chapters_text[3][:200]}...")

Preview of Chapter 1: Zesenveertigste hoofdstuk.
Maston ten tooneele.
Er was een groote beweging aan boord van de Susquehanna. Officieren en matrozen vergaten het vreeslijk gevaar dat zij hadden
geloopen, verbrijzeld en na...


In [ ]:
# @title 3. Initialize Voice Engine
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=False,
)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

💡 PRO TIP: To record your own voice, you can use free tools like
Audacity (Desktop) or an online voice recorder (like online-voice-recorder.com).

Simply record yourself reading the text below, save it as a .wav file, and upload it here.

*"Typisch, zuchtte de vrolijke wiskundeleraar, terwijl hij naar het grijze schoolbord wees."*

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

# Note: These sentences are phonetically optimized to cover a wide range
# of sounds, which helps the model create a more accurate voice clone.
ref_text = "Typisch, zuchtte de vrolijke wiskundeleraar, terwijl hij naar het grijze schoolbord wees."

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

Upload a reference audio file (wav/mp3/flac):


Saving Record (online-audio-converter.com).wav to Record (online-audio-converter.com) (2).wav
Uploaded: Record (online-audio-converter.com) (2).wav


In [ ]:
# @title 4. Generate Audiobook (Parallel Processing)
BOOK_PREFIX = "Jules Verne - De Reis naar de Maan"
OUTPUT_DIR = "output_chapters"
THREADS = 2 # Adjust based on GPU memory
os.makedirs(OUTPUT_DIR, exist_ok=True)

def sanitize_filename(text):
    return re.sub(r'[^a-zA-Z0-9]', '_', text)[:50]

def process_chapter(chapter_data):
    idx, content = chapter_data
    title = content.split('\n')[0]
    safe_name = f"{idx:03d}_{sanitize_filename(title)}"
    wav_path = os.path.join(OUTPUT_DIR, f"{safe_name}.wav")
    mp3_path = os.path.join(OUTPUT_DIR, f"{safe_name}.mp3")

    try:
        # Using 'male' as a valid instruction item instead of 'natural narration'
        audio = model.generate(text=content, instruct="male", ref_audio=ref_audio_path, ref_text=ref_text)
        sf.write(wav_path, audio[0], 24000)
        subprocess.run(['ffmpeg', '-y', '-i', wav_path, '-b:a', '64k', mp3_path], capture_output=True)
        if os.path.exists(wav_path): os.remove(wav_path)
        print(f"Done: {safe_name}")
    except Exception as e: print(f"Error in {safe_name}: {e}")

print(f"Starting generation...")
with ThreadPoolExecutor(max_workers=THREADS) as executor:
    executor.map(process_chapter, list(enumerate(chapters_text)))

shutil.make_archive("audiobook_package", 'zip', OUTPUT_DIR)
files.download("audiobook_package.zip")

Starting generation...


Enjoy listening!